In [1]:
#feature engineering
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import datetime 
from datetime import timedelta

warnings.filterwarnings('ignore')



In [2]:
customer=pd.read_excel(r'C:\first_data_science_proj\Customer_Segmentation--UCI-\data\raw\online+retail\Online Retail.xlsx')


In [3]:
df=customer.copy()
df['InvoiceDate']=pd.to_datetime(df['InvoiceDate'])
df['TotalAmount']=df['Quantity']*df['UnitPrice']



In [4]:
print(f' data loaded')
print(f' transactions: {len(df):,}')
print(f' customers: {df["CustomerID"].nunique():,}')
print(f' date range: {df["InvoiceDate"].min().date()} to {df["InvoiceDate"].max().date()}')


 data loaded
 transactions: 541,909
 customers: 4,372
 date range: 2010-12-01 to 2011-12-09


In [5]:
df_clean=df.dropna(subset=['CustomerID']).copy()
print(f' after removing transactions with missing customer id: {df_clean.shape[0]:,}rows')

df_clean=df_clean[~df_clean['InvoiceNo'].astype(str).str.startswith('C')].copy()
print(f' after removing cancelled transactions: {df_clean.shape[0]:,}rows')

df_clean=df_clean[(df_clean['Quantity']>0)& (df_clean['UnitPrice']>0)].copy()
print(f' after removing negative values : {df_clean.shape[0]:,}rows')

for col in ['Quantity','UnitPrice','TotalAmount']:
    upper=df_clean[col].quantile(0.995)
    df_clean=df_clean[df_clean[col]<=upper].copy()

print(f' after outlier capping: {df_clean.shape[0]:,}rows')
print(f"final unique customers: {df_clean['CustomerID'].nunique():,}")

reference_date=df_clean['InvoiceDate'].max()+timedelta(days=1)
print(f' reference date: {reference_date.date()}')



 after removing transactions with missing customer id: 406,829rows
 after removing cancelled transactions: 397,924rows
 after removing negative values : 397,884rows
 after outlier capping: 392,263rows
final unique customers: 4,289
 reference date: 2011-12-10


In [10]:
df_clean['CustomerID'].loc[:10]
# or


0     17850.0
1     17850.0
2     17850.0
3     17850.0
4     17850.0
5     17850.0
6     17850.0
7     17850.0
8     17850.0
9     13047.0
10    13047.0
Name: CustomerID, dtype: float64

In [11]:
df_clean['CustomerID'].duplicated().sum()

np.int64(387974)

In [12]:
len(df_clean['CustomerID'])

392263

In [7]:
customer_features={}
for cust_id in df_clean['CustomerID'].unique():
    cust_data=df_clean[df_clean['CustomerID']==cust_id].sort_values('InvoiceDate')

    customer_features[cust_id]={
        'Recency':(reference_date-cust_data['InvoiceDate'].max()).days,
        'Frequency': cust_data['InvoiceNo'].nunique(),
        'Monetary': cust_data['TotalAmount'].sum(),
    }

print(f" sample-customer1000: r={customer_features[10000]['Recency']} days,"
      f" f={customer_features[10000]['Frequency']} orders,"
      f"m={customer_features[10000]['Monetary']:.2f}")



KeyError: 10000

In [6]:
customer_features={}
for cust_id in df_clean['CustomerID'].unique():
    cust_data=df_clean[df_clean['CustomerID']==cust_id].sort_values('InvoiceDate')

    customer_features[cust_id]={
        'Recency':(reference_date-cust_data['InvoiceDate'].max()).days,
        'Frequency':cust_data['InvoiceNo'].nunique(),
        'Monetary': cust_data['TotalAmount'].sum(),
    }

print(f' core rfm features calculated for all customers')
sample_id = next(iter(customer_features))
sample = customer_features[sample_id]
print(f" sample - customer {sample_id} r={sample['Recency']} days",
      f" f={sample['Frequency']} orders",
      f" m={sample['Monetary']:.2f}")

 core rfm features calculated for all customers
 sample - customer 17850.0 r=372 days  f=34 orders  m=4783.46


In [ ]:
for cust_id in df_clean['CustomerID'].unique():
    cust_data=df_clean[df_clean['CustomerID']==cust_id]

    order_totals=cust_data.groupby('InvoiceNo')['TotalAmount'].sum()

    customer_features[cust_id].update({
        'AvgOrderValue':order_totals.mean(),
        'MaxOrderValue':order_totals.max(),
        'MinOrderValue': order_totals.min(),
    })

print(f'sample-customer 10000:')